In [ ]:
!pip install -r requirements.txt

In [ ]:
# If the assignment takes too long, you may want to download the code and run it locally to take advantage of your GPU. 
# Package Imports
import numpy as np
import tensorflow as tf
import gymnasium as gym
from collections import deque
import random

# Environment setup and Variables
try:
    env = gym.make("CartPole-v1", render_mode=None)
except Exception:
    print('Failed to initialize environment! Make sure that gymnasium was installed correctly!')
    sys.exit(1)

state_shape = env.observation_space.shape[0]
action_size = env.action_space.n

# Hyperparameters
GAMMA = 0.99
EXPLORATION_MAX = 1.0
EXPLORATION_MIN = 0.01
EXPLORATION_DECAY = 0.990
LEARNING_RATE = 0.001
BATCH_SIZE = 64
TRAIN_START = 1000
MEMORY_SIZE = 2000
EPISODES = 300

# Counters during training
train_freq = 4
step_count = 0
target_update_freq = 10
# If an error occurs below, it is because the environment is looking for a GPU.

In [ ]:
memory = deque(maxlen=MEMORY_SIZE)

# DQN Builder
def build_dqn_model():
    try:
        model = tf.keras.Sequential([
            tf.keras.layers.Input(shape=(state_shape,)),
            tf.keras.layers.Dense(64, activation='relu'),
            tf.keras.layers.Dense(64, activation='relu'),
            tf.keras.layers.Dense(action_size, activation='linear')
        ])
        model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE), loss='mse')
        return model
    except Exception:
        print('Error occurred while building the DQN model!')
        sys.exit(1)

In [ ]:
# Calling the build_dqn_model method
model = build_dqn_model()
target_model = build_dqn_model()
target_model.set_weights(model.get_weights())

# If an error displays, it is simply trying to connect to the GPU

In [ ]:
# Random Action selection for exploration
def get_action(state, EXPLORATION_MAX):
    try:
        if np.random.rand() <= EXPLORATION_MAX:
            return random.randrange(action_size)
        q_values = model.predict(np.array([state]), verbose=0) #Remove verbose=0 to see every prediction display
        return np.argmax(q_values[0])
    except Exception:
            print("Error during get_action method!")
            sys.exit(1)

In [ ]:
# Experience Replay
def experience_replay():
    if len(memory) < TRAIN_START:
        return

    batch = random.sample(memory, BATCH_SIZE)
    states = np.zeros((BATCH_SIZE, state_shape))
    next_states = np.zeros((BATCH_SIZE, state_shape))
    actions, rewards, completions = [], [], []

    try:
        for i, (state, action, reward, state_next, terminal) in enumerate(batch):
            states[i] = state
            next_states[i] = state_next
            actions.append(action)
            rewards.append(reward)
            completions.append(terminal)

        target_q = model.predict(states, verbose=0) #Remove verbose=0 to see every prediction display
        next_q = target_model.predict(next_states, verbose=0) #Remove verbose=0 to see every prediction display

        for i in range(BATCH_SIZE):
            if completions[i]:
                target_q[i][actions[i]] = rewards[i]
            else:
                target_q[i][actions[i]] = rewards[i] + GAMMA * np.max(next_q[i])

        model.fit(states, target_q, batch_size=BATCH_SIZE, verbose=0)
    except Exception:
        print("Error during replay")
        sys.exit(1)

In [ ]:
# A list to keep track of all of the episodes
episode_rewards = []

# Main loop
# Will iterate through based on the number of EPISODES originally listed up top
for e in range(EPISODES):
    state, _ = env.reset()
    done = False
    total_reward = 0

    while not done:
        action = get_action(state, EXPLORATION_MAX)
        next_state, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated
        
        reward = reward if not done else -10

        memory.append((state, action, reward, next_state, done))
        state = next_state
        total_reward += reward
        
        step_count += 1
        if step_count % train_freq == 0:
            experience_replay()

    if EXPLORATION_MAX > EXPLORATION_MIN:
        EXPLORATION_MAX *= EXPLORATION_DECAY
        EXPLORATION_MAX = max(EXPLORATION_MIN, EXPLORATION_MAX)

    if e % target_update_freq == 0:
        target_model.set_weights(model.get_weights())

    episode_rewards.append(total_reward)
    # This will display the average reward of every 10 episodes occurring.
    # Comment this section and add the below print method if you want to display every episode
    if e % 10 == 0:
        avg_reward = np.mean(episode_rewards[-10:])
        print(f"Episode {e+1}: Average Reward = {avg_reward:.2f}, Exploration = {EXPLORATION_MAX:.3f}")
    
    # Logic to complete execution if average is greater than 195
    if len(episode_rewards) >= 100:
        avg_last_hun = np.mean(episode_rewards[-100:])
        if avg_last_hun >= 195:
            print(f"Solved at episode {e}: Average reward over last 100: {avg_last_hun:.2f}")
            break
        
    # Comment this in if you want to see every iteration of the training occurring     
    # print(f"Episode {e+1}: Total Reward = {total_reward}, Exploration = {EXPLORATION_MAX:.3f}")

# Saving model as needed    
model.save("Cartpole_model.h5")
print("Training complete! Model saved as Cartpole_model")

# Note, if an error displays, it is attempting to connect to the GPU.
# It will not connect and run on CPU. 
# Code will take some time to run, which is commonplace for real life models.
# Make sure your computer does not go to sleep, and take a well-deserved break!

In [ ]:
# Make your observations here as it pertains to the assignment rubric.